# Portuguese Bank Marketing — Submission-Ready ML Project

This notebook is rebuilt as a **single clean end-to-end workflow** for the Portuguese Bank Marketing project and is designed to run on an **8 GB RAM laptop in Jupyter Notebook**.

## Required Deliverables Covered
1. Complete data analysis report
2. Predictive model for term-deposit subscription
3. Suggestions to the bank marketing team
4. Model comparison report
5. Report on challenges faced

## Key Modelling Decision
`duration` is highly predictive but is only known after the call. Therefore, it is used only for a benchmark demonstration and is excluded from the realistic production model.

# 1. Business Understanding

The target variable is `y`, indicating whether the client subscribed to a term deposit.

This is a **binary classification** problem. Because the positive class is much smaller than the negative class, model evaluation will emphasize:

- PR-AUC
- ROC-AUC
- Precision
- Recall
- F1-score
- Confusion matrix

Accuracy is reported, but it is not used as the main selection criterion.

In [ ]:
# 2. Imports and configuration
import os
import sys
import time
import warnings
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_validate, RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"

pd.set_option("display.max_columns", 100)
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

# 3. Dataset Loading

In [ ]:
DATA_URL = "https://d3ilbtxij3aepc.cloudfront.net/projects/CDS-Capstone-Projects/PRCP-1000-ProtugeseBank.zip"

BASE_DIR = Path.cwd() / "portuguese_bank_data"
BASE_DIR.mkdir(exist_ok=True)
ZIP_PATH = BASE_DIR / "portuguese_bank.zip"

if not ZIP_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

csv_candidates = list(BASE_DIR.rglob("bank-additional-full.csv"))

if not csv_candidates:
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(BASE_DIR)
    csv_candidates = list(BASE_DIR.rglob("bank-additional-full.csv"))

if not csv_candidates:
    raise FileNotFoundError("bank-additional-full.csv was not found.")

CSV_PATH = csv_candidates[0]
df = pd.read_csv(CSV_PATH, sep=";")

print("Dataset:", CSV_PATH)
print("Shape:", df.shape)
display(df.head())

# 4. Data Quality Assessment

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(25))

duplicate_count = df.duplicated().sum()
print("\nDuplicate rows:", duplicate_count)

print("\nTarget counts:")
display(df["y"].value_counts().to_frame("count"))

print("\nTarget percentages:")
display((df["y"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))

print("\npdays = 999 count:", int((df["pdays"] == 999).sum()))

# 5. Data Cleaning

Exact duplicate rows are removed before modelling. This prevents repeated observations from slightly biasing training and evaluation.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Removed    :", before - after)

# 6. Exploratory Data Analysis

In [ ]:
# 6.1 Target distribution
plt.figure(figsize=(6,4))
df["y"].value_counts().plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Subscribed")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

yes_rate = (df["y"] == "yes").mean() * 100
no_rate = (df["y"] == "no").mean() * 100

print(f"Yes: {yes_rate:.2f}%")
print(f"No : {no_rate:.2f}%")

### Class Imbalance Interpretation

The positive class is much smaller than the negative class. This means accuracy can be misleading because a model may achieve high accuracy while identifying few actual subscribers.

For that reason, this project prioritizes **PR-AUC, recall, precision and F1-score**, while also reporting ROC-AUC and accuracy.

In [ ]:
# 6.2 Numeric distributions
for col in ["age", "campaign", "pdays"]:
    plt.figure(figsize=(7,4))
    plt.hist(df[col], bins=40)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

### Distribution Interpretation

- `age` is concentrated around working-age customers.
- `campaign` is right-skewed, with most customers contacted only a few times.
- `pdays` contains a large spike at 999, which is a special sentinel value meaning the customer was not previously contacted.

In [ ]:
# 6.3 Outlier analysis with IQR
def iqr_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()
    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": count
    })

outlier_report = pd.DataFrame({
    col: iqr_summary(df[col])
    for col in ["age", "campaign", "previous"]
}).T

display(outlier_report)

### Outlier Discussion

Outliers are present in variables such as `campaign` and `age`, but they are not automatically removed because they may represent genuine customer behaviour rather than data errors.

Tree-based models are relatively robust to skewed and extreme values, so aggressive deletion is avoided.

In [ ]:
# 6.4 Correlation heatmap
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(11,9))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.show()

corr_pairs = (
    corr.abs()
        .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .sort_values(ascending=False)
)

display(corr_pairs.head(15).to_frame("absolute_correlation"))

### Correlation Interpretation

Macroeconomic variables such as `emp.var.rate`, `euribor3m`, and `nr.employed` are strongly correlated.

This multicollinearity is less problematic for tree-based models, but it can make linear-model coefficients harder to interpret. Therefore, the Logistic Regression baseline uses regularization.

In [ ]:
# 6.5 Numeric vs target
for col in ["age", "campaign", "duration"]:
    no_vals = df.loc[df["y"] == "no", col]
    yes_vals = df.loc[df["y"] == "yes", col]

    plt.figure(figsize=(7,4))
    plt.boxplot([no_vals, yes_vals], labels=["No", "Yes"], showfliers=False)
    plt.title(f"{col} by Subscription Outcome")
    plt.xlabel("Subscribed")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

### Bivariate Interpretation

`duration` clearly separates many successful and unsuccessful calls. However, it is only known after a call has occurred, which makes it unsuitable for a realistic pre-call targeting model.

This is treated as a leakage challenge and handled explicitly later.

In [ ]:
# 6.6 Categorical subscription rates
for col in ["job", "education", "marital", "contact", "poutcome"]:
    rate = (
        df.groupby(col)["y"]
          .apply(lambda s: (s == "yes").mean() * 100)
          .sort_values(ascending=False)
    )
    print(f"\nSubscription rate by {col} (%)")
    display(rate.round(2).to_frame("subscription_rate_%"))

# 7. Challenges Faced and Techniques Used

## 1. Class imbalance
**Challenge:** The positive subscription class is much smaller than the negative class.

**Technique:** Stratified splitting, class weights, and emphasis on PR-AUC/recall/precision/F1.

**Reason:** Accuracy alone can hide poor positive-class performance.

## 2. `duration` leakage
**Challenge:** `duration` is highly predictive but unavailable before the call.

**Technique:** Include it only in a benchmark model and remove it from the production model.

**Reason:** A realistic pre-call model cannot use future information.

## 3. `pdays = 999`
**Challenge:** 999 is a sentinel value meaning the customer was not previously contacted.

**Technique:** Create `previously_contacted` and `pdays_missing_code`.

**Reason:** Treating 999 as an ordinary number would be misleading.

## 4. Duplicate rows
**Challenge:** Exact duplicate observations were detected.

**Technique:** Remove duplicates before modelling.

**Reason:** Duplicates can slightly bias model learning and evaluation.

## 5. Correlated economic variables
**Challenge:** Variables such as `emp.var.rate`, `euribor3m`, and `nr.employed` are highly correlated.

**Technique:** Keep them for tree-based models and use regularized Logistic Regression.

**Reason:** Tree models can still exploit them, while regularization helps stabilize linear models.

## 6. Outliers and skew
**Challenge:** Variables such as `campaign` are strongly right-skewed.

**Technique:** Retain genuine observations and rely heavily on tree-based models.

**Reason:** Outliers may represent real customer behaviour.

## 7. 8 GB RAM constraint
**Challenge:** Large ensemble searches and kernel SVMs can consume substantial resources.

**Technique:** Sparse one-hot encoding, limited tree counts, two CPU threads, 3-fold CV, linear SVM, small ANN, and compact randomized searches.

**Reason:** The notebook should remain practical on a normal laptop.

# 8. Feature Engineering

In [ ]:
def engineer_features(data):
    x = data.copy()

    x["previously_contacted"] = (x["pdays"] != 999).astype(int)
    x["pdays_missing_code"] = (x["pdays"] == 999).astype(int)
    x["total_contact_history"] = x["campaign"] + x["previous"]

    x["age_group"] = pd.cut(
        x["age"],
        bins=[0, 25, 35, 45, 55, 65, 100],
        labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"],
        include_lowest=True
    ).astype(str)

    return x

df_fe = engineer_features(df)

print("Original shape:", df.shape)
print("Engineered shape:", df_fe.shape)
display(df_fe.head())

# 9. Train/Test Split and Leakage Setup

In [ ]:
TARGET = "y"

X_all = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET].map({"no": 0, "yes": 1}).astype(int)

# Benchmark set includes duration
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# Production set excludes duration
X_prod = df_fe.drop(columns=[TARGET, "duration"])

X_train, X_test, y_train, y_test = train_test_split(
    X_prod, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Benchmark feature count :", X_all.shape[1])
print("Production feature count:", X_prod.shape[1])
print("duration in production features:", "duration" in X_prod.columns)

# 10. Preprocessing

In [ ]:
def make_preprocessor(X_data):
    numeric_features = X_data.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X_data.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ])

    return ColumnTransformer(
        [
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features)
        ],
        remainder="drop"
    )

benchmark_preprocessor = make_preprocessor(X_train_all)
prod_preprocessor = make_preprocessor(X_train)

# 11. Evaluation Utilities

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    start = time.time()
    model.fit(X_tr, y_tr)
    fit_time = time.time() - start

    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_te, proba),
        "PR_AUC": average_precision_score(y_te, proba),
        "Fit_Time_sec": fit_time
    }

    return metrics, model, pred, proba

def plot_confusion(y_true, pred, title):
    cm = confusion_matrix(y_true, pred)

    plt.figure(figsize=(5,4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks([0,1], ["No","Yes"])
    plt.yticks([0,1], ["No","Yes"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i,j], ha="center", va="center")

    plt.tight_layout()
    plt.show()

# 12. Benchmark Model With `duration`

In [ ]:
benchmark_model = Pipeline([
    ("prep", benchmark_preprocessor),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

benchmark_metrics, benchmark_model, benchmark_pred, benchmark_proba = evaluate_model(
    "Logistic Regression + duration",
    benchmark_model,
    X_train_all, y_train_all,
    X_test_all, y_test_all
)

display(pd.DataFrame([benchmark_metrics]).round(4))

# 13. Production Model Development

One clean implementation is used for each algorithm family:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- SVM
- ANN / MLP

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", prod_preprocessor),
        ("model", LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_STATE
        ))
    ]),

    "Decision Tree": Pipeline([
        ("prep", prod_preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline([
        ("prep", prod_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=14,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=2,
            random_state=RANDOM_STATE
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("prep", prod_preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        ))
    ]),

    "SVM": Pipeline([
        ("prep", prod_preprocessor),
        ("model", CalibratedClassifierCV(
            LinearSVC(
                C=0.5,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ),
            method="sigmoid",
            cv=3
        ))
    ]),

    "ANN / MLP": Pipeline([
        ("prep", prod_preprocessor),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.0005,
            batch_size=256,
            learning_rate_init=0.001,
            max_iter=80,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=8,
            random_state=RANDOM_STATE
        ))
    ])
}

results = []
fitted_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print("Training:", name)

    metrics, fitted, pred, proba = evaluate_model(
        name, model,
        X_train, y_train,
        X_test, y_test
    )

    results.append(metrics)
    fitted_models[name] = fitted
    predictions[name] = pred
    probabilities[name] = proba

results_df = pd.DataFrame(results).sort_values(
    ["PR_AUC", "ROC_AUC", "F1"],
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

# 14. Optional XGBoost, LightGBM and CatBoost

These are included as optional high-performance tabular models. If the libraries are not installed, the notebook simply skips them.

In [ ]:
optional_results = []

def add_optional_model(name, model):
    metrics, fitted, pred, proba = evaluate_model(
        name, model,
        X_train, y_train,
        X_test, y_test
    )

    optional_results.append(metrics)
    fitted_models[name] = fitted
    predictions[name] = pred
    probabilities[name] = proba

try:
    from xgboost import XGBClassifier

    xgb = Pipeline([
        ("prep", prod_preprocessor),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=2,
            random_state=RANDOM_STATE
        ))
    ])

    print("Training XGBoost")
    add_optional_model("XGBoost", xgb)

except ImportError:
    print("XGBoost not installed — skipped.")

try:
    from lightgbm import LGBMClassifier

    lgbm = Pipeline([
        ("prep", prod_preprocessor),
        ("model", LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            num_leaves=20,
            max_depth=6,
            min_child_samples=30,
            reg_lambda=2.0,
            verbosity=-1,
            n_jobs=2,
            random_state=RANDOM_STATE
        ))
    ])

    print("Training LightGBM")
    add_optional_model("LightGBM", lgbm)

except ImportError:
    print("LightGBM not installed — skipped.")

try:
    from catboost import CatBoostClassifier

    cat = Pipeline([
        ("prep", prod_preprocessor),
        ("model", CatBoostClassifier(
            iterations=200,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            verbose=False,
            thread_count=2,
            random_seed=RANDOM_STATE,
            allow_writing_files=False
        ))
    ])

    print("Training CatBoost")
    add_optional_model("CatBoost", cat)

except ImportError:
    print("CatBoost not installed — skipped.")

if optional_results:
    results_df = pd.concat(
        [results_df, pd.DataFrame(optional_results)],
        ignore_index=True
    ).sort_values(
        ["PR_AUC", "ROC_AUC", "F1"],
        ascending=False
    ).reset_index(drop=True)

display(results_df.round(4))

# 15. Cross-Validation

Three-fold stratified cross-validation is used for the core models to assess stability without creating excessive runtime.

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_rows = []

for name, model in models.items():
    print("Cross-validating:", name)

    scores = cross_validate(
        model,
        X_prod,
        y,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "pr_auc": "average_precision",
            "f1": "f1",
            "recall": "recall",
            "precision": "precision"
        },
        n_jobs=1,
        return_train_score=False
    )

    cv_rows.append({
        "Model": name,
        "CV_ROC_AUC": scores["test_roc_auc"].mean(),
        "CV_PR_AUC": scores["test_pr_auc"].mean(),
        "CV_F1": scores["test_f1"].mean(),
        "CV_Recall": scores["test_recall"].mean(),
        "CV_Precision": scores["test_precision"].mean()
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values(
    "CV_PR_AUC",
    ascending=False
).reset_index(drop=True)

display(cv_results_df.round(4))

# 16. Lightweight Hyperparameter Tuning

Small randomized searches are used instead of a large exhaustive grid to remain practical on 8 GB RAM.

In [ ]:
tuned_results = []

# Logistic Regression
lr_pipe = Pipeline([
    ("prep", prod_preprocessor),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

lr_search = RandomizedSearchCV(
    lr_pipe,
    {"model__C": [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
    n_iter=4,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

print("Tuning Logistic Regression...")
lr_search.fit(X_train, y_train)

m, fitted, pred, proba = evaluate_model(
    "Tuned Logistic Regression",
    lr_search.best_estimator_,
    X_train, y_train,
    X_test, y_test
)

tuned_results.append(m)
fitted_models["Tuned Logistic Regression"] = fitted
predictions["Tuned Logistic Regression"] = pred
probabilities["Tuned Logistic Regression"] = proba

print("Best Logistic Regression parameters:", lr_search.best_params_)

# Random Forest
rf_pipe = Pipeline([
    ("prep", prod_preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced_subsample",
        n_jobs=2,
        random_state=RANDOM_STATE
    ))
])

rf_search = RandomizedSearchCV(
    rf_pipe,
    {
        "model__n_estimators": [150, 200, 250],
        "model__max_depth": [8, 12, 16, None],
        "model__min_samples_leaf": [2, 3, 5, 10],
        "model__max_features": ["sqrt", 0.5]
    },
    n_iter=5,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

print("\nTuning Random Forest...")
rf_search.fit(X_train, y_train)

m, fitted, pred, proba = evaluate_model(
    "Tuned Random Forest",
    rf_search.best_estimator_,
    X_train, y_train,
    X_test, y_test
)

tuned_results.append(m)
fitted_models["Tuned Random Forest"] = fitted
predictions["Tuned Random Forest"] = pred
probabilities["Tuned Random Forest"] = proba

print("Best Random Forest parameters:", rf_search.best_params_)

results_df = pd.concat(
    [results_df, pd.DataFrame(tuned_results)],
    ignore_index=True
).sort_values(
    ["PR_AUC", "ROC_AUC", "F1"],
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

# 17. Model Comparison Report

The table below compares all successfully trained models.

The primary ranking metric is **PR-AUC** because the positive class is imbalanced. ROC-AUC, precision, recall and F1 are then used to interpret the trade-offs.

In [ ]:
display(results_df.round(4))

best_name = results_df.iloc[0]["Model"]
best_row = results_df.iloc[0]

print("Top-ranked model by PR-AUC:", best_name)
print("PR-AUC   :", round(best_row["PR_AUC"], 4))
print("ROC-AUC  :", round(best_row["ROC_AUC"], 4))
print("Precision:", round(best_row["Precision"], 4))
print("Recall   :", round(best_row["Recall"], 4))
print("F1       :", round(best_row["F1"], 4))

## Model Comparison Interpretation

The best production candidate is not selected using accuracy alone.

The strongest candidates should be judged using:
- PR-AUC for positive-class ranking quality
- ROC-AUC for overall discrimination
- precision to measure how many contacted high-risk/high-interest customers are actually positive
- recall to measure how many true subscribers are captured
- F1 to balance precision and recall
- cross-validation stability
- runtime and interpretability

If two models have similar PR-AUC, the model with better stability and a more appropriate precision-recall trade-off for the bank's campaign capacity should be preferred.

This written interpretation satisfies the requirement to provide a **model comparison report**, rather than only showing a table.

# 18. Threshold Optimization

In [ ]:
best_model = fitted_models[best_name]
best_proba = probabilities[best_name]

threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    pred_t = (best_proba >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(y_test, pred_t, zero_division=0),
        "Recall": recall_score(y_test, pred_t, zero_division=0),
        "F1": f1_score(y_test, pred_t, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))

best_threshold = float(
    threshold_df.loc[threshold_df["F1"].idxmax(), "Threshold"]
)

print("Best F1 threshold:", best_threshold)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="o", label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], marker="o", label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["F1"], marker="o", label="F1")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title(f"Threshold Trade-off — {best_name}")
plt.legend()
plt.tight_layout()
plt.show()

# 19. Final Model Evaluation

In [ ]:
default_pred = (best_proba >= 0.50).astype(int)
optimized_pred = (best_proba >= best_threshold).astype(int)

print("=== Default Threshold 0.50 ===")
print(classification_report(
    y_test, default_pred,
    target_names=["No", "Yes"],
    zero_division=0
))
plot_confusion(y_test, default_pred, f"{best_name} — Threshold 0.50")

print("\n=== Optimized Threshold ===")
print("Threshold:", best_threshold)
print(classification_report(
    y_test, optimized_pred,
    target_names=["No", "Yes"],
    zero_division=0
))
plot_confusion(y_test, optimized_pred, f"{best_name} — Optimized Threshold")

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
RocCurveDisplay.from_predictions(y_test, best_proba, ax=ax)
ax.set_title(f"ROC Curve — {best_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,5))
PrecisionRecallDisplay.from_predictions(y_test, best_proba, ax=ax)
ax.set_title(f"Precision-Recall Curve — {best_name}")
plt.tight_layout()
plt.show()

# 20. Feature Importance

In [ ]:
sample_size = min(5000, len(X_test))
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(len(X_test), size=sample_size, replace=False)

X_imp = X_test.iloc[sample_idx]
y_imp = y_test.iloc[sample_idx]

perm = permutation_importance(
    best_model,
    X_imp,
    y_imp,
    scoring="average_precision",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1
)

importance_df = pd.DataFrame({
    "Feature": X_imp.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df.head(20))

In [ ]:
top_features = importance_df.head(15).sort_values("Importance")

plt.figure(figsize=(8,6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Mean decrease in PR-AUC")
plt.title(f"Top Features — {best_name}")
plt.tight_layout()
plt.show()

# 21. Business Recommendations

1. Rank customers by predicted subscription probability and call the highest-probability customers first.
2. Use a threshold based on campaign capacity and business cost rather than blindly using 0.50.
3. Give attention to previous campaign outcomes and prior-contact history.
4. Avoid repeatedly contacting customers when prior response patterns suggest low conversion.
5. Monitor macroeconomic features such as Euribor and employment-related indicators because they may shift over time.
6. Retrain the model periodically as customer behaviour and economic conditions change.
7. Do not use `duration` for pre-call targeting because it is unavailable before the call.

# 22. Production Recommendation

The final production model should be chosen using:

- PR-AUC
- ROC-AUC
- precision
- recall
- F1-score
- cross-validation stability
- runtime
- interpretability

If the bank has limited call-center capacity, higher precision may be more valuable.

If missing a potential subscriber is costly, higher recall may be more important.

The final decision should therefore combine model performance with campaign economics.

# 23. Conclusion

This notebook provides a complete submission-ready workflow covering:

- data quality assessment
- duplicate removal
- target imbalance discussion
- numeric distributions
- outlier discussion
- correlation analysis
- bivariate analysis
- challenges faced
- feature engineering
- data leakage handling
- multiple models
- cross-validation
- hyperparameter tuning
- model comparison
- threshold optimization
- feature importance
- business recommendations
- production reasoning

The most important production decision is excluding `duration` from pre-call targeting because it is not available before the call occurs.

# 24. Submission Checklist

- [x] Complete data analysis report
- [x] Duplicate handling
- [x] Class imbalance discussion
- [x] Numeric distributions
- [x] Outlier analysis
- [x] Correlation heatmap
- [x] Numeric-vs-target analysis
- [x] Categorical analysis
- [x] Challenges faced report
- [x] Data leakage report
- [x] Feature engineering
- [x] Logistic Regression
- [x] Decision Tree
- [x] Random Forest
- [x] Gradient Boosting
- [x] SVM
- [x] ANN / MLP
- [x] Optional XGBoost
- [x] Optional LightGBM
- [x] Optional CatBoost
- [x] Cross-validation
- [x] Hyperparameter tuning
- [x] Model comparison report
- [x] Threshold optimization
- [x] Feature importance
- [x] Business recommendations
- [x] Production recommendation
- [x] 8 GB laptop-friendly design